# TNG300-1: large-scale environment and $z=0$ descendant halo mass

This notebook asks whether the three-dimensional environment of $z\simeq8$, $M_{200c}\sim10^{11}M_\odot$ halos contains descendant-mass information not supplied by halo mass alone. It reuses the existing descendant catalog and does **not** repeat merger-tree tracking.

All final masses are the snapshot-99 FoF-host `Group_M_Crit200` values inherited from the preceding analysis. “Conditional width” below means the width of a simulation-derived conditional distribution, not a formal Bayesian posterior.

## Cell 1: imports

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial import cKDTree
from tqdm.auto import tqdm
from IPython.display import display, Markdown

import illustris_python as il

plt.rcParams.update({
    "figure.figsize": (7.2, 4.8),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11,
})

## Cell 2: simulation path and parameters

TNG positions are comoving kpc/$h$ and catalog masses are $10^{10}M_\odot/h$. The notebook converts positions to cMpc and masses to physical $M_\odot$. The initial halo-neighbor threshold, $10^{9.5}M_\odot$, corresponds to roughly tens of TNG300-1 dark-matter particles and should be varied in a resolution-sensitivity test.

In [ ]:
base_candidates = [
    Path("/home/tnguser/sims.TNG/L205n2500TNG/output"),
    Path("../sims.TNG/TNG300-1/output").resolve(),
    Path("../sims.TNG/L205n2500TNG/output").resolve(),
]
basePath = next((p for p in base_candidates if p.is_dir()), None)
if basePath is None:
    raise FileNotFoundError(
        "TNG300-1 was not found. Set basePath to its output directory. Tried:\n"
        + "\n".join(str(p) for p in base_candidates)
    )

descendant_csv = Path.cwd() / "tng300_z8_M11_descendants.csv"
environment_csv = Path.cwd() / "tng300_z8_M11_descendants_environment.csv"

snap_z8 = 8
snap_z0 = 99
logM_min = 10.8
logM_max = 11.2
R_list_cMpc = np.array([1, 2, 3, 5, 8, 10, 15, 20], dtype=float)
logMstar_threshold = 8.0
logMhalo_threshold = 9.5
require_subhalo_flag = True
exclude_target = True
query_chunk_size = 256
random_state = 42
min_conditional_n = 30
sfr_floor = 1.0e-3

header_z8 = il.groupcat.loadHeader(str(basePath), snap_z8)
z_z8 = float(header_z8["Redshift"])
h = float(header_z8["HubbleParam"])
box_size_ckpc_h = float(header_z8["BoxSize"])
box_size_cMpc = box_size_ckpc_h / (1000.0 * h)

assert np.isclose(h, 0.6774, atol=1e-4, rtol=0)
assert np.all(np.diff(R_list_cMpc) > 0) and R_list_cMpc[-1] < box_size_cMpc / 2

print(f"basePath              : {basePath}")
print(f"snapshot {snap_z8} redshift : {z_z8:.6f}")
print(f"snapshot z=0          : {snap_z0}")
print(f"h                     : {h:.4f}")
print(f"box size              : {box_size_cMpc:.3f} cMpc")
print(f"environment radii     : {R_list_cMpc.tolist()} cMpc")
print(f"galaxy threshold      : logMstar > {logMstar_threshold}")
print(f"halo threshold        : logM200c > {logMhalo_threshold}")

## Cell 3: load the existing descendant catalog

Rows with failed tracking or non-finite required values are removed. The narrow $z\simeq8$ mass selection is reapplied defensively.

In [ ]:
required_columns = [
    "GroupID_z8", "SubhaloID_z8", "M200c_z8", "logM200c_z8",
    "GroupID_z0", "SubhaloID_z0", "M200c_z0", "logM200c_z0",
]
if not descendant_csv.is_file():
    raise FileNotFoundError(f"Missing {descendant_csv}; run the descendant notebook first.")

raw_desc = pd.read_csv(descendant_csv)
missing = sorted(set(required_columns) - set(raw_desc.columns))
if missing:
    raise ValueError(f"Descendant catalog is missing columns: {missing}")

finite = raw_desc[required_columns].notna().all(axis=1)
mass_cut = (raw_desc["logM200c_z8"] > logM_min) & (raw_desc["logM200c_z8"] < logM_max)
env = raw_desc.loc[finite & mass_cut, required_columns].copy().reset_index(drop=True)
for column in ["GroupID_z8", "SubhaloID_z8", "GroupID_z0", "SubhaloID_z0"]:
    env[column] = env[column].astype(np.int64)

print(f"Descendant-catalog rows : {len(raw_desc):,}")
print(f"Finite tracked rows     : {finite.sum():,}")
print(f"Environment targets     : {len(env):,}")
display(env.head())

## Cell 4: target-halo positions at $z\simeq8$

`GroupPos` is converted from comoving kpc/$h$ to cMpc by division by $1000h$. Positions are wrapped into $[0,L)$ for periodic-tree compatibility.

In [ ]:
halo_catalog = il.groupcat.loadHalos(
    str(basePath), snap_z8, fields=["GroupPos", "Group_M_Crit200"]
)
group_pos_ckpc_h = np.asarray(halo_catalog["GroupPos"], dtype=float)
group_m200c_msun = np.asarray(halo_catalog["Group_M_Crit200"], dtype=float) * 1.0e10 / h
group_pos_cMpc = np.mod(group_pos_ckpc_h / (1000.0 * h), box_size_cMpc)

target_group_ids = env["GroupID_z8"].to_numpy(dtype=np.int64)
if target_group_ids.min() < 0 or target_group_ids.max() >= len(group_pos_cMpc):
    raise IndexError("A target GroupID_z8 is outside the snapshot-8 FoF catalog.")
target_pos_cMpc = group_pos_cMpc[target_group_ids]

catalog_target_mass = group_m200c_msun[target_group_ids]
mass_difference_dex = np.abs(np.log10(catalog_target_mass) - env["logM200c_z8"].to_numpy())
if np.nanmax(mass_difference_dex) > 1e-4:
    raise ValueError("Target GroupID-to-M200c consistency check failed.")

print(f"Snapshot-8 FoF halos loaded : {len(group_pos_cMpc):,}")
print(f"Target positions             : {len(target_pos_cMpc):,}")
print(f"Maximum mass mismatch        : {np.nanmax(mass_difference_dex):.3e} dex")

## Cell 5: snapshot-8 galaxy catalog

In the TNG six-component mass vector, PartType4 (column 4) is stars/winds. The richness catalog uses total bound stellar mass above $10^8M_\odot$ and, by default, `SubhaloFlag == 1` to reject non-cosmological fragments. `SubhaloSFR` is loaded for the later individual-galaxy comparison.

In [ ]:
subhalo_catalog = il.groupcat.loadSubhalos(
    str(basePath), snap_z8,
    fields=["SubhaloPos", "SubhaloMassType", "SubhaloGrNr", "SubhaloFlag", "SubhaloSFR"],
)
subhalo_pos_cMpc = np.mod(
    np.asarray(subhalo_catalog["SubhaloPos"], dtype=float) / (1000.0 * h),
    box_size_cMpc,
)
subhalo_mass_type = np.asarray(subhalo_catalog["SubhaloMassType"], dtype=float)
subhalo_grnr = np.asarray(subhalo_catalog["SubhaloGrNr"], dtype=np.int64)
subhalo_flag = np.asarray(subhalo_catalog["SubhaloFlag"], dtype=bool)
subhalo_sfr = np.asarray(subhalo_catalog["SubhaloSFR"], dtype=float)

stellar_parttype_index = 4
subhalo_mstar_msun = subhalo_mass_type[:, stellar_parttype_index] * 1.0e10 / h
galaxy_mask = np.isfinite(subhalo_mstar_msun) & (subhalo_mstar_msun > 10**logMstar_threshold)
if require_subhalo_flag:
    galaxy_mask &= subhalo_flag

galaxy_ids = np.flatnonzero(galaxy_mask)
galaxy_pos_cMpc = subhalo_pos_cMpc[galaxy_ids]
galaxy_mstar_msun = subhalo_mstar_msun[galaxy_ids]

target_subhalo_ids = env["SubhaloID_z8"].to_numpy(dtype=np.int64)
if target_subhalo_ids.min() < 0 or target_subhalo_ids.max() >= len(subhalo_pos_cMpc):
    raise IndexError("A target SubhaloID_z8 is outside the snapshot-8 subhalo catalog.")
if not np.all(subhalo_grnr[target_subhalo_ids] == target_group_ids):
    raise ValueError("Target central subhalo-to-FoF mapping is inconsistent.")

env["MstarCentral_z8"] = subhalo_mstar_msun[target_subhalo_ids]
env["logMstarCentral_z8"] = np.where(
    env["MstarCentral_z8"] > 0, np.log10(env["MstarCentral_z8"]), np.nan
)
env["SFRCentral_z8"] = subhalo_sfr[target_subhalo_ids]
env["logSFRCentral_z8"] = np.log10(env["SFRCentral_z8"] + sfr_floor)

print(f"All snapshot-8 subhalos : {len(subhalo_pos_cMpc):,}")
print(f"Selected galaxies       : {len(galaxy_ids):,}")
print(f"Require SubhaloFlag     : {require_subhalo_flag}")

## Cell 6: snapshot-8 neighbor-halo catalog

Neighbor halos are FoF groups above the parameterized $M_{200c}$ threshold. The target group is excluded later by its global Group ID, whether or not it passes the neighbor threshold.

In [ ]:
halo_neighbor_mask = np.isfinite(group_m200c_msun) & (group_m200c_msun > 10**logMhalo_threshold)
halo_neighbor_ids = np.flatnonzero(halo_neighbor_mask)
halo_neighbor_pos_cMpc = group_pos_cMpc[halo_neighbor_ids]
halo_neighbor_mass_msun = group_m200c_msun[halo_neighbor_ids]

print(f"FoF halos above threshold : {len(halo_neighbor_ids):,}")
print(f"Threshold                 : M200c > 10^{logMhalo_threshold:g} Msun")
print(f"Fraction of all FoF groups: {len(halo_neighbor_ids)/len(group_m200c_msun):.3%}")

## Cell 7: periodic-distance function and boundary test

The minimum-image convention replaces each Cartesian separation by the shorter of $\Delta x$ and $L-\Delta x$. A pair separated across the box boundary is used as a unit test.

In [ ]:
def periodic_distance(pos1, pos2, boxsize):
    pos1 = np.asarray(pos1, dtype=float)
    pos2 = np.asarray(pos2, dtype=float)
    delta = np.abs(pos1 - pos2)
    delta = np.minimum(delta, boxsize - delta)
    return np.sqrt(np.sum(delta**2, axis=-1))


edge_a = np.array([0.2, 10.0, 20.0])
edge_b = np.array([box_size_cMpc - 0.2, 10.0, 20.0])
edge_distance = periodic_distance(edge_a, edge_b, box_size_cMpc)
assert np.isclose(edge_distance, 0.4)

test_tree = cKDTree(np.vstack([edge_a, edge_b]), boxsize=box_size_cMpc)
assert set(test_tree.query_ball_point(edge_a, r=0.5)) == {0, 1}
print(f"Periodic edge test passed: distance = {edge_distance:.3f} cMpc")

## Cell 8: galaxy-environment measurement function

The generic engine queries only the largest radius in chunks. For each target it computes periodic distances once, sorts once, and obtains cumulative counts and weighted sums at all radii. Global object IDs guarantee that the target central is removed exactly rather than by coordinate matching.

In [ ]:
def cumulative_periodic_environment(
    target_positions,
    catalog_positions,
    catalog_weights,
    radii,
    boxsize,
    catalog_global_ids,
    target_global_ids,
    chunk_size=256,
    description="environment",
):
    target_positions = np.asarray(target_positions, dtype=float)
    catalog_positions = np.asarray(catalog_positions, dtype=float)
    catalog_weights = np.asarray(catalog_weights, dtype=float)
    radii = np.asarray(radii, dtype=float)
    catalog_global_ids = np.asarray(catalog_global_ids, dtype=np.int64)
    target_global_ids = np.asarray(target_global_ids, dtype=np.int64)

    if len(catalog_positions) != len(catalog_weights) or len(catalog_positions) != len(catalog_global_ids):
        raise ValueError("Catalog positions, weights, and IDs must have equal lengths.")
    if len(target_positions) != len(target_global_ids):
        raise ValueError("Target positions and IDs must have equal lengths.")

    tree = cKDTree(catalog_positions, boxsize=boxsize)
    counts = np.zeros((len(target_positions), len(radii)), dtype=np.int64)
    weighted_sums = np.zeros((len(target_positions), len(radii)), dtype=float)

    starts = range(0, len(target_positions), chunk_size)
    for start in tqdm(starts, total=(len(target_positions) + chunk_size - 1) // chunk_size,
                      desc=description):
        stop = min(start + chunk_size, len(target_positions))
        try:
            neighbor_lists = tree.query_ball_point(
                target_positions[start:stop], r=float(radii[-1]), workers=-1
            )
        except TypeError:
            neighbor_lists = tree.query_ball_point(
                target_positions[start:stop], r=float(radii[-1])
            )

        for local_i, neighbor_idx in enumerate(neighbor_lists):
            target_i = start + local_i
            idx = np.asarray(neighbor_idx, dtype=np.int64)
            if idx.size == 0:
                continue

            if exclude_target:
                idx = idx[catalog_global_ids[idx] != target_global_ids[target_i]]
            if idx.size == 0:
                continue

            delta = np.abs(catalog_positions[idx] - target_positions[target_i])
            delta = np.minimum(delta, boxsize - delta)
            distances = np.sqrt(np.sum(delta**2, axis=1))
            order = np.argsort(distances)
            distances = distances[order]
            weights_sorted = catalog_weights[idx][order]
            cumulative_weights = np.cumsum(weights_sorted)

            k = np.searchsorted(distances, radii, side="left")  # strict distance < R
            counts[target_i] = k
            nonzero = k > 0
            weighted_sums[target_i, nonzero] = cumulative_weights[k[nonzero] - 1]

    return counts, weighted_sums


def measure_galaxy_environment():
    return cumulative_periodic_environment(
        target_pos_cMpc, galaxy_pos_cMpc, galaxy_mstar_msun,
        R_list_cMpc, box_size_cMpc, galaxy_ids, target_subhalo_ids,
        chunk_size=query_chunk_size, description="Galaxy environment",
    )

## Cell 9: halo-environment measurement function

The identical engine is used for FoF neighbors, with `Group_M_Crit200` as the weight. The target FoF group is removed by `GroupID_z8`.

In [ ]:
def measure_halo_environment():
    return cumulative_periodic_environment(
        target_pos_cMpc, halo_neighbor_pos_cMpc, halo_neighbor_mass_msun,
        R_list_cMpc, box_size_cMpc, halo_neighbor_ids, target_group_ids,
        chunk_size=query_chunk_size, description="Halo environment",
    )

## Cell 10: measure all target environments

`Ngal`, `MstarTot`, `Nhalo`, and `MhaloTot` are cumulative within each radius and exclude the target central/group. `deltaGal` uses the full-box number density and is therefore an ideal three-dimensional overdensity, not an observationally projected estimator.

In [ ]:
ngal, mstar_tot = measure_galaxy_environment()
nhalo, mhalo_tot = measure_halo_environment()

def radius_tag(radius):
    return f"{radius:g}".replace(".", "p")


galaxy_number_density = len(galaxy_ids) / box_size_cMpc**3
for j, radius in enumerate(R_list_cMpc):
    tag = radius_tag(radius)
    env[f"Ngal_R{tag}"] = ngal[:, j]
    env[f"MstarTot_R{tag}"] = mstar_tot[:, j]
    env[f"Nhalo_R{tag}"] = nhalo[:, j]
    env[f"MhaloTot_R{tag}"] = mhalo_tot[:, j]
    expected_random = galaxy_number_density * (4.0 * np.pi / 3.0) * radius**3
    env[f"deltaGal_R{tag}"] = env[f"Ngal_R{tag}"] / expected_random - 1.0

assert np.all(np.diff(ngal, axis=1) >= 0)
assert np.all(np.diff(nhalo, axis=1) >= 0)
assert np.all(np.diff(mstar_tot, axis=1) >= -1e-6)
assert np.all(np.diff(mhalo_tot, axis=1) >= -1e-6)
print("All cumulative environment quantities are monotonic with radius.")
display(env.head())

## Cell 11: save the environment catalog

In [ ]:
env.to_csv(environment_csv, index=False)
print(f"Saved {len(env):,} rows and {len(env.columns)} columns to:")
print(environment_csv)

## Cell 12: basic environment distributions

Mass sums are plotted as $\log_{10}(M+1)$ so that empty apertures remain visible. The printed quantiles provide a quick outlier and dynamic-range check.

In [ ]:
diagnostic_columns = ["Ngal_R5", "Ngal_R10", "MstarTot_R5", "MhaloTot_R5"]
display(env[diagnostic_columns].describe(percentiles=[0.01, 0.16, 0.5, 0.84, 0.99]))

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
plot_specs = [
    ("Ngal_R5", False), ("Ngal_R10", False),
    ("MstarTot_R5", True), ("MhaloTot_R5", True),
]
for ax, (column, log_transform) in zip(axes.flat, plot_specs):
    values = np.log10(env[column] + 1.0) if log_transform else env[column]
    ax.hist(values, bins=30, alpha=0.82)
    ax.set(xlabel=(f"log10({column} + 1)" if log_transform else column), ylabel="Targets")
plt.tight_layout()
plt.show()

## Cell 13: correlation with descendant mass at every scale

Spearman coefficients are used because the environment observables are non-Gaussian and counts contain ties. Constant inputs return `NaN` rather than raising an exception.

In [ ]:
def safe_spearman(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    if valid.sum() < 3 or np.unique(x[valid]).size < 2 or np.unique(y[valid]).size < 2:
        return np.nan, np.nan, int(valid.sum())
    rho, p_value = spearmanr(x[valid], y[valid])
    return float(rho), float(p_value), int(valid.sum())


correlation_rows = []
for radius in R_list_cMpc:
    tag = radius_tag(radius)
    row = {"R_cMpc": radius}
    for prefix in ["Ngal", "MstarTot", "Nhalo", "MhaloTot"]:
        rho, p_value, n = safe_spearman(env[f"{prefix}_R{tag}"], env["logM200c_z0"])
        row[f"rho_{prefix}"] = rho
        row[f"p_{prefix}"] = p_value
        row[f"N_{prefix}"] = n
    correlation_rows.append(row)
correlation_table = pd.DataFrame(correlation_rows)
display(correlation_table.round(4))

## Cell 14: scale dependence and best radius

The “best” scale maximizes $|\rho|$ over the tested radius grid; the signed coefficient is retained. Choosing the best radius on the same sample introduces a look-elsewhere/selection effect, so this is exploratory rather than an unbiased performance estimate.

In [ ]:
observable_labels = {
    "Ngal": r"$N_{\rm gal}$",
    "MstarTot": r"$M_{\star,\rm tot}$",
    "Nhalo": r"$N_{\rm halo}$",
    "MhaloTot": r"$M_{\rm halo,tot}$",
}
best_environment = {}

fig, ax = plt.subplots(figsize=(7.5, 5.0))
for prefix, label in observable_labels.items():
    rho_values = correlation_table[f"rho_{prefix}"].to_numpy()
    ax.plot(R_list_cMpc, rho_values, "o-", label=label)
    if np.all(~np.isfinite(rho_values)):
        best_environment[prefix] = (np.nan, np.nan)
    else:
        idx = int(np.nanargmax(np.abs(rho_values)))
        best_environment[prefix] = (float(R_list_cMpc[idx]), float(rho_values[idx]))
        print(f"{prefix:10s}: best R = {R_list_cMpc[idx]:g} cMpc, rho = {rho_values[idx]:+.3f}")
ax.axhline(0, color="k", lw=1)
ax.set(xlabel="R [cMpc]", ylabel=r"Spearman $\rho$ with $\log M_0$", xticks=R_list_cMpc)
ax.legend(ncol=2)
plt.show()

best_ngal_radius, best_ngal_rho = best_environment["Ngal"]
best_ngal_tag = radius_tag(best_ngal_radius)

## Cell 15: baseline halo-mass correlation

The main comparison remains restricted to $10^{10.8}<M_8/M_\odot<10^{11.2}$. No wider auxiliary sample is introduced here.

In [ ]:
rho_m8, p_m8, n_m8 = safe_spearman(env["logM200c_z8"], env["logM200c_z0"])
print(f"rho(logM8, logM0) = {rho_m8:+.3f} (p={p_m8:.3e}, N={n_m8})")
for prefix, (radius, rho) in best_environment.items():
    comparison = "higher" if abs(rho) > abs(rho_m8) else "not higher"
    print(f"Best |rho| for {prefix} is {comparison} than the narrow-range M8 baseline.")

## Cell 16: richness scatter plots at representative scales

Rank-balanced bins are used only to draw median and 16–84 percentile trends; the individual points retain the original discrete richness values.

In [ ]:
def binned_percentile_trend(x, y, n_bins=8):
    temp = pd.DataFrame({"x": x, "y": y}).dropna()
    temp["bin"] = pd.qcut(temp["x"].rank(method="first"), q=n_bins, labels=False)
    return temp.groupby("bin").agg(
        x=("x", "median"), y=("y", "median"),
        y16=("y", lambda v: np.percentile(v, 16)),
        y84=("y", lambda v: np.percentile(v, 84)),
        N=("y", "size"),
    )


fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
for ax, radius in zip(axes, [2, 5, 10]):
    tag = radius_tag(radius)
    x = env[f"Ngal_R{tag}"]
    y = env["logM200c_z0"]
    trend = binned_percentile_trend(x, y)
    rho, _, _ = safe_spearman(x, y)
    ax.scatter(x, y, s=10, alpha=0.22, edgecolors="none")
    ax.errorbar(
        trend["x"], trend["y"],
        yerr=[trend["y"] - trend["y16"], trend["y84"] - trend["y"]],
        fmt="o-", color="k", capsize=2,
    )
    ax.set(xlabel=rf"$N_{{\rm gal}}(<{radius}\,\rm cMpc)$", title=rf"$\rho={rho:.3f}$")
axes[0].set_ylabel(r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$")
plt.tight_layout()
plt.show()

## Cell 17: conditional descendant distributions at the best richness scale

The discrete richness variable is divided into rank-balanced tertiles. Ties at a boundary can therefore be split deterministically; the actual richness ranges and sample counts are shown and should be considered when interpreting the curves.

In [ ]:
def summarize_distribution(values, label="sample"):
    values = np.asarray(pd.Series(values).dropna(), dtype=float)
    if values.size == 0:
        return pd.Series({"Label": label, "N": 0, "Median": np.nan, "P16": np.nan,
                          "P84": np.nan, "W68": np.nan})
    p16, median, p84 = np.percentile(values, [16, 50, 84])
    return pd.Series({"Label": label, "N": values.size, "Median": median,
                      "P16": p16, "P84": p84, "W68": p84 - p16})


def rank_tertiles(series):
    return pd.qcut(series.rank(method="first"), q=3, labels=["low", "middle", "high"])


best_ngal_column = f"Ngal_R{best_ngal_tag}"
env["Ngal_best_tertile"] = rank_tertiles(env[best_ngal_column])
tertile_rows = []
for label, group in env.groupby("Ngal_best_tertile", observed=True):
    row = summarize_distribution(group["logM200c_z0"], str(label))
    row["Ngal_min"] = group[best_ngal_column].min()
    row["Ngal_max"] = group[best_ngal_column].max()
    tertile_rows.append(row)
ngal_tertile_stats = pd.DataFrame(tertile_rows)

common_bins = np.linspace(env["logM200c_z0"].min(), env["logM200c_z0"].max(), 27)
fig, ax = plt.subplots()
for label, group in env.groupby("Ngal_best_tertile", observed=True):
    ax.hist(group["logM200c_z0"], bins=common_bins, density=True, histtype="step", lw=2,
            label=f"{label} environment (N={len(group)})")
ax.set(xlabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$", ylabel="Probability density")
ax.legend()
plt.show()
display(ngal_tertile_stats.round(3))

## Cell 18: baseline versus environment-conditioned width

For a single observed tertile, the relevant row is its own $W_{68}$. As an overall summary of the partition, the notebook also reports the sample-count-weighted mean within-tertile width. A narrower selected distribution is not automatically a calibrated posterior improvement.

In [ ]:
baseline_stats = summarize_distribution(env["logM200c_z0"], "M8 only")
ngal_tertile_stats["DeltaW68_vs_baseline"] = baseline_stats["W68"] - ngal_tertile_stats["W68"]
weighted_conditional_w68 = np.average(
    ngal_tertile_stats["W68"], weights=ngal_tertile_stats["N"]
)

print(f"Baseline W68                         : {baseline_stats['W68']:.3f} dex")
print(f"Weighted mean within-tertile W68     : {weighted_conditional_w68:.3f} dex")
print(f"Baseline - weighted conditional W68 : {baseline_stats['W68'] - weighted_conditional_w68:+.3f} dex")
display(ngal_tertile_stats.round(3))

## Cell 19: comparison with individual central-galaxy properties

Central stellar mass and SFR were loaded in Cell 5, so this comparison does not depend on an additional saved catalog. Environment entries use each observable's exploratory best scale.

In [ ]:
property_rows = []
comparison_features = {
    "M8": "logM200c_z8",
    "central Mstar": "logMstarCentral_z8",
    "central SFR": "logSFRCentral_z8",
}
for prefix, (radius, _) in best_environment.items():
    comparison_features[f"{prefix} (R={radius:g} cMpc)"] = f"{prefix}_R{radius_tag(radius)}"

for label, column in comparison_features.items():
    rho, p_value, n = safe_spearman(env[column], env["logM200c_z0"])
    property_rows.append({"Observable": label, "N": n, "Spearman rho": rho, "p-value": p_value})
property_comparison = pd.DataFrame(property_rows).sort_values(
    "Spearman rho", key=np.abs, ascending=False
)
display(property_comparison.round({"Spearman rho": 3, "p-value": 5}))

## Cell 20: environment information after removing the linear $M_8$ trend

A simple least-squares line is fitted on the full narrow-range sample, and environment correlations are recomputed against $\Delta\log M_0$. This is a descriptive residual-information test, not a causal or fully cross-validated partial-correlation analysis.

In [ ]:
linear_coeff = np.polyfit(env["logM200c_z8"], env["logM200c_z0"], deg=1)
env["logM0_pred_from_M8"] = np.polyval(linear_coeff, env["logM200c_z8"])
env["DeltaLogM0_at_fixed_M8"] = env["logM200c_z0"] - env["logM0_pred_from_M8"]

residual_rows = []
for radius in R_list_cMpc:
    tag = radius_tag(radius)
    row = {"R_cMpc": radius}
    for prefix in observable_labels:
        rho, p_value, n = safe_spearman(env[f"{prefix}_R{tag}"], env["DeltaLogM0_at_fixed_M8"])
        row[f"rho_resid_{prefix}"] = rho
    residual_rows.append(row)
residual_correlation_table = pd.DataFrame(residual_rows)
display(residual_correlation_table.round(4))

for prefix in observable_labels:
    values = residual_correlation_table[f"rho_resid_{prefix}"].to_numpy()
    idx = int(np.nanargmax(np.abs(values)))
    print(f"{prefix:10s}: residual best R={R_list_cMpc[idx]:g} cMpc, rho={values[idx]:+.3f}")

## Cell 21: optional predictive regression

Random Forests provide an auxiliary nonlinear comparison. The split is grouped by `GroupID_z0`, preventing multiple $z\simeq8$ targets with the same final FoF host from leaking across train and test. All models use the same rows and split. The main scientific result remains the transparent conditional analysis above.

In [ ]:
regression_available = False
regression_results = pd.DataFrame()

try:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.model_selection import GroupShuffleSplit
    from sklearn.metrics import mean_squared_error, mean_absolute_error

    best_mstar_radius = best_environment["MstarTot"][0]
    best_mstar_column = f"MstarTot_R{radius_tag(best_mstar_radius)}"
    env["logMstarTot_best"] = np.log10(env[best_mstar_column] + 1.0)

    model_features = {
        "M8 only": ["logM200c_z8"],
        "M8 + Ngal": ["logM200c_z8", best_ngal_column],
        "M8 + Ngal + MstarTot": ["logM200c_z8", best_ngal_column, "logMstarTot_best"],
    }
    all_features = model_features["M8 + Ngal + MstarTot"]
    reg_data = env.dropna(subset=all_features + ["logM200c_z0", "GroupID_z0"])

    if len(reg_data) < 500:
        print(f"Skipping regression: only {len(reg_data)} complete rows (<500).")
    else:
        splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=random_state)
        train_pos, test_pos = next(splitter.split(reg_data, groups=reg_data["GroupID_z0"]))
        train_idx = reg_data.index[train_pos]
        test_idx = reg_data.index[test_pos]

        rows = []
        for label, features in model_features.items():
            model = RandomForestRegressor(
                n_estimators=400, min_samples_leaf=10, max_features=1.0,
                n_jobs=-1, random_state=random_state,
            )
            model.fit(reg_data.loc[train_idx, features], reg_data.loc[train_idx, "logM200c_z0"])
            prediction = model.predict(reg_data.loc[test_idx, features])
            truth = reg_data.loc[test_idx, "logM200c_z0"]
            rows.append({
                "Model": label, "N_train": len(train_idx), "N_test": len(test_idx),
                "RMSE_dex": np.sqrt(mean_squared_error(truth, prediction)),
                "MAE_dex": mean_absolute_error(truth, prediction),
            })
        regression_results = pd.DataFrame(rows)
        regression_available = True
        display(regression_results.round(3))
except ImportError as exc:
    print(f"scikit-learn unavailable; optional regression skipped ({exc}).")

## Cell 22: environment-scale summary

In [ ]:
scale_summary_rows = []
for prefix, label in observable_labels.items():
    radius, rho = best_environment[prefix]
    row_at_best = correlation_table.loc[correlation_table["R_cMpc"] == radius].iloc[0]
    scale_summary_rows.append({
        "Observable": prefix,
        "Best_R_cMpc": radius,
        "Spearman_rho": rho,
        "N": int(row_at_best[f"N_{prefix}"]),
    })
scale_summary = pd.DataFrame(scale_summary_rows)
display(scale_summary.round(3))

## Cell 23: main summary figure

The left panel shows scale dependence for all environment statistics. The right panel shows descendant-mass distributions in best-scale richness tertiles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for prefix, label in observable_labels.items():
    axes[0].plot(R_list_cMpc, correlation_table[f"rho_{prefix}"], "o-", label=label)
axes[0].axhline(rho_m8, color="k", ls="--", label=rf"$M_8$ baseline ($\rho={rho_m8:.2f}$)")
axes[0].set(xlabel="R [cMpc]", ylabel=r"Spearman $\rho$ with $\log M_0$", xticks=R_list_cMpc)
axes[0].legend(fontsize=8, ncol=2)

for label, group in env.groupby("Ngal_best_tertile", observed=True):
    axes[1].hist(group["logM200c_z0"], bins=common_bins, density=True,
                 histtype="step", lw=2, label=f"{label} Ngal (N={len(group)})")
axes[1].set(xlabel=r"$\log_{10}(M_{200c,z=0}^{\rm host}/M_\odot)$", ylabel="Probability density",
            title=rf"Best richness scale: {best_ngal_radius:g} cMpc")
axes[1].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Cell 24: automatically generated interpretation

Run the final code cell to render a Markdown summary from the measured values. The interpretation distinguishes simulation-derived conditional widths from calibrated observational posteriors.

In [ ]:
strongest_row = scale_summary.iloc[np.nanargmax(np.abs(scale_summary["Spearman_rho"]))]
best_ngal_high = ngal_tertile_stats.loc[ngal_tertile_stats["Label"] == "high"].iloc[0]

central_mstar_rho = property_comparison.loc[
    property_comparison["Observable"] == "central Mstar", "Spearman rho"
].iloc[0]
central_sfr_rho = property_comparison.loc[
    property_comparison["Observable"] == "central SFR", "Spearman rho"
].iloc[0]
environment_beats_galaxy = abs(strongest_row["Spearman_rho"]) > max(abs(central_mstar_rho), abs(central_sfr_rho))

if regression_available:
    baseline_rmse = regression_results.loc[regression_results["Model"] == "M8 only", "RMSE_dex"].iloc[0]
    full_rmse = regression_results.loc[regression_results["Model"] == "M8 + Ngal + MstarTot", "RMSE_dex"].iloc[0]
    regression_text = (
        f"The optional grouped holdout test changed RMSE from {baseline_rmse:.3f} dex "
        f"to {full_rmse:.3f} dex when Ngal and total stellar mass were added."
    )
else:
    regression_text = "The optional regression was unavailable and is not used in the conclusions."

display(Markdown(fr"""
### Interpretation

1. For **{len(env):,}** halos at $z={z_z8:.3f}$, the baseline descendant distribution has
   median $\log_{{10}}(M_0/M_\odot)={baseline_stats['Median']:.3f}$ and $W_{{68}}={baseline_stats['W68']:.3f}$ dex.
2. Within the narrow initial mass range, halo mass alone has Spearman $\rho={rho_m8:+.3f}$ with descendant mass.
3. Galaxy richness is strongest at **{best_ngal_radius:g} cMpc**, where $\rho={best_ngal_rho:+.3f}$.
   The high-richness tertile has median $\log M_0={best_ngal_high['Median']:.3f}$ and $W_{{68}}={best_ngal_high['W68']:.3f}$ dex.
4. The weighted mean within-richness-tertile width is **{weighted_conditional_w68:.3f} dex**, a change of
   **{baseline_stats['W68'] - weighted_conditional_w68:+.3f} dex** relative to the unconditioned baseline.
5. Across all tested environment definitions, **{strongest_row['Observable']}** at **{strongest_row['Best_R_cMpc']:g} cMpc**
   has the largest absolute rank correlation, $\rho={strongest_row['Spearman_rho']:+.3f}$.
6. The strongest environment statistic {'does' if environment_beats_galaxy else 'does not'} exceed both central stellar mass
   ($\rho={central_mstar_rho:+.3f}$) and central SFR ($\rho={central_sfr_rho:+.3f}$) in absolute rank correlation.
7. {regression_text}
8. Correlation is not causation, the best-scale selection is exploratory, and small-scale results depend on subhalo finding and
   mass completeness. Large scales approach the box mean and may dilute predictive information.
9. A JWST-facing extension should replace the ideal 3D apertures with projected cylinders, photometric-redshift selection,
   stellar-mass/flux completeness, interlopers, masks, and measurement-error forward modeling.

These results quantify simulation-derived conditional distributions, not a formal Bayesian descendant-mass posterior.
"""))